16b. Dựng đồ thị NF-ToN-IoT-v3 với cửa sổ W = 1, 2, 10 giây, CPU (task 5.5, GĐ G)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = False
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])
assert ALLOW_GPU or INFO["device"] == "CPU", "Chưa được phép dùng GPU: tắt GPU trong cài đặt notebook"

## Dựng đồ thị NF-ToN-IoT-v3 với W = S = 1, 2, 10 giây
Cùng cách chia với W = 5 của notebook 04: `blocked`, 10 khối, theo node (`balance=1`), `gap = 1`. Mỗi lần dựng chạy trong một
tiến trình riêng (RAM khoảng 19 GB, tiến trình kết thúc thì giải phóng). Chỉ dựng đồ thị và chia tập, **chưa huấn luyện gì**
và không đọc nhãn test cho bất kỳ quyết định nào.

In [ ]:
WINDOWS = [10, 2, 1]
OUT = f"{WORK}/processed"
INFO["windows"] = WINDOWS
csv = find("NF-ToN-IoT-v3.csv")
codes = {}
for w in WINDOWS:
    ds = f"nf_ton_iot_v3_w{w}"
    with step(f"build_{ds}"):
        r = run([sys.executable, "-I", "scripts/build_dataset_cli.py", "nf_ton_iot_v3", csv, str(w), str(w), "blocked", "1",
                 f"{OUT}/{ds}", "blocks=10", "balance=1", f"tag=nf_ton_iot_v3_w{w}s{w}_blocked"], ok=(0, 3), cwd=REPO)
        codes[ds] = r.returncode
INFO["exit_codes"] = codes
save_info()
run(["du", "-sh", OUT])

## Tóm tắt: số node mỗi lớp mỗi tập, kiểm tra, kích thước file

In [ ]:
rows = {}
for w in WINDOWS:
    ds = f"nf_ton_iot_v3_w{w}"
    s = json.load(open(f"{OUT}/{ds}/summary.json", encoding="utf-8"))
    rows[ds] = {"n_graphs": s["n_graphs"], "checks_ok": all(c["ok"] for c in s["checks"]),
                "failed_checks": [c["name"] for c in s["checks"] if not c["ok"]],
                "split_counts": s["split_counts"], "files_mb": {k: round(v / 2**20, 1) for k, v in s["files"].items()},
                "fingerprint": s["fingerprint_sha256"]}
    print(ds, json.dumps(rows[ds], ensure_ascii=False, indent=1))
INFO["summary"] = rows
save_info()
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())